<a href="https://colab.research.google.com/github/LampOfSocrates/reid/blob/main/EEEM071_CourseWork2604_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This insures that 2 seconda after editing a module it gets reloaded without restarting the kernel
%load_ext autoreload
%autoreload 2

# Step 1: GPU Selection
1. Find "Edit" tab above, select "Hardware accelerator" and choose "GPU"
2. Run bellow command to check what GPU you got

In [2]:
from datetime import datetime

NOTEBOOK_START_TIME = datetime.now()

In [3]:
!nvidia-smi

Sun Apr 26 22:50:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
| 30%   40C    P8             25W /  300W |    1541MiB /  16303MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Step 2: Code Preparation

We need to maintain our codebase with git history, so a file system (Google Drive) is needed
1. Select the left file icon and mount your Google Drive
2. Move path to Google Drive
3. Git clone code base


In [4]:
%%time

from pathlib import Path
import sys
import os

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False

if is_colab():
  from google.colab import drive
  drive.mount('/content/drive')

  %cd /content
  #!pip install -U --no-cache-dir gdown --pre

  # please download datasets from assignment doc link and upload, then unzip it.

  # Copy the uploaded zip from Google Drive into the Colab runtime, then unzip it locally.
  if not os.path.exists('/content/VeRi'):
      print('Unzipping VeRi dataset...')
      !cp "/content/drive/MyDrive/Colab Notebooks/data/VeRi.zip" /content/VeRi.zip
      !unzip /content/VeRi.zip -d /content/
  else:
      print('VeRi dataset already exists. Skipping unzip.')
  %ls -l /content

  # code
  if not os.path.exists('/content/reid'):
      print('Cloning repository...')
      !git clone https://github.com/LampOfSocrates/reid /content/reid
  else:
      print('Repository already exists. Pulling latest changes...')
      %cd /content/reid
      !git pull
  %cd /content/reid
  %pip install -r requirements_colab.txt

  repo_root = Path('/content/reid')
  data_root = Path('/content')

else: # Local run 
    repo_root = Path.cwd()
    data_root = Path(r'C:\Users\soura\Code\2026\reid\data')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'is_colab={is_colab()}')
print(f'repo_root={repo_root}')
print(f'data_root={data_root}')

is_colab=False
repo_root=C:\Users\soura\code\2026\reid
data_root=C:\Users\soura\Code\2026\reid\data
CPU times: total: 0 ns
Wall time: 1.04 ms


In [5]:
if is_colab():
    from IPython.display import Javascript, display
    display(Javascript("""
        setInterval(() => {
            const btn = document.querySelector('#connect');
            if (btn) btn.click();
        }, 60000);
    """))
    print("Keep-alive started: clicking reconnect every 60 s")

# Step 3: Data Preparation

Because reading images from Google Drive is very slow, we download datasets to Colab temporary file
1. Install gdown
2. Download data
3. Unzip data with password

 Note that we have a folder called /content/drive/MyDrive/Veri with the full data

# Set Globals

6 model runs take 30 minutes with DATA_FRACTION at 1% and MAX_EPOCH at 10 . vgg16 doesnt even respond to that.

Let this be the checked in combination


In [6]:
from coursework.src.utils.metrics_report import render_run, compare_runs, export_html
from datetime import datetime

if is_colab(): #8 / 20 run 
    MAX_EPOCH = 20
    DATA_FRACTION = 1
else: 
    MAX_EPOCH = 3  # only used if model specific max_epoch is not set 
    DATA_FRACTION = 0.001
RUNTIME = datetime.now().strftime("%d%m_%H%M")


In [7]:
# Defaults shared by all experiments. Anything in BASE_CFG can be overridden
# per-experiment via the dict below.
BASE_CFG = {
    "optim": "amsgrad",
    "lr": "0.0003",
    "train_batch_size": 128,
    "test_batch_size": 100,
    "workers": 8,
    "stepsize": "20 40",
    "eval_freq": 1,
    "patience": 10,
}

DRIVE_SYNC = "/content/drive/MyDrive/Colab Notebooks/reid/logs" if is_colab() else ""

# Each entry: experiment name -> {arch + any overrides that differ from BASE_CFG}
# Hyperparameters drawn from the original papers in coursework/papers/ (see knowledge.md sec 8)
SECTION_1_MODELS = {
    # Swin-T -- Li et al., Array 16 (2022) 100255: 60 epochs, batch 32, input 256x256.
    # Paper omits lr/optim; falling back to BoT 3.5e-4 Adam + 10-epoch warmup since main.py uses Adam.
    # K=4 imgs/ID matches TransReID transformer recipe (default in args.py).
    "swin_t_custom": {
        "arch": "swin_t_custom",
        "lr": "3.5e-4",
        "max_epoch": "60",
        "lr_scheduler": "multi_step_warmup",
        "warmup_epochs": 10,
    },
    "swin_t_fc512": {
        "arch": "swin_t_fc512",
        "lr": "3.5e-4",
        "max_epoch": "60",
        "lr_scheduler": "multi_step_warmup",
        "warmup_epochs": 10,
    },
    # CLIP-SENet v1 dual -- arXiv 2502.16815 sec IV.B.1: ADAM, lr 5e-4, cosine, 24 epochs,
    # batch 128 = 16 IDs x 8 imgs/ID -> num_instances=8.
    "clip_senet_v1_dual": {
        "arch": "clip_senet_v1_dual",
        "lr": "5e-4",
        "max_epoch": "24",
        "lr_scheduler": "cosine",
        "num_instances": 8,
    },
    # CLIP-SENet v3 -- same paper recipe (24 epochs cosine, lr 5e-4, K=8); was 2-epoch smoke test.
    "clip_senet_v3_ibn_supcon": {
        "arch": "clip_senet_v3_ibn_supcon",
        "lr": "5e-4",
        "max_epoch": "24",
        "lr_scheduler": "cosine",
        "num_instances": 8,
    },
    #"mobilenet_v3_small": {"arch": "mobilenet_v3_small"},
    #"resnet50_fc512":     {"arch": "resnet50_fc512"},
    # CLIP-SENet v2 frozen -- backbone frozen, only AFEM+head trainable -> bump lr ~2x.
    # "clip_senet_v2_frozen": {"arch": "clip_senet_v2_frozen", "lr": "1e-3", "max_epoch": "24", "lr_scheduler": "cosine", "num_instances": 8},  # disabled
}


def run_experiment(name, cfg, question="Q1", extra_args=()):
    cfg = {**BASE_CFG, **cfg}  # per-experiment overrides win
    arch = cfg["arch"]
    save_dir = f"logs/{RUNTIME}/{question}/{name}-veri"
    max_epoch = cfg.get("max_epoch", MAX_EPOCH)  # cfg override > global default
    scheduler_flags = ""
    if "lr_scheduler" in cfg:
        scheduler_flags += f"--lr-scheduler {cfg['lr_scheduler']} "
    if cfg.get("warmup_epochs"):
        scheduler_flags += f"--warmup-epochs {cfg['warmup_epochs']} "
    if cfg.get("num_instances"):
        scheduler_flags += f"--num-instances {cfg['num_instances']} "
    drive_sync_flag = f"--drive-sync-dir \"{DRIVE_SYNC}\" " if DRIVE_SYNC else ""
    cmd = (
        f"python coursework/main.py "
        f"-s veri -t veri -a {arch} "
        f"--experiment {name} "
        f"--root {data_root} "
        f"--height 224 --width 224 "
        f"--optim {cfg['optim']} --lr {cfg['lr']} "
        f"--max-epoch {max_epoch} "
        f"--stepsize {cfg['stepsize']} "
        f"--eval-freq {cfg['eval_freq']} "
        f"--patience {cfg['patience']} "
        f"--train-batch-size {cfg['train_batch_size']} "
        f"--test-batch-size {cfg['test_batch_size']} "
        f"--workers {cfg['workers']} "
        f"--data-fraction {DATA_FRACTION} "
        f"--train-sampler RandomIdentitySampler "
        f"{scheduler_flags}"
        f"{drive_sync_flag}"
        f"--save-dir {save_dir} "
        + " ".join(extra_args)
    )
    print()
    print(f"=== {name}  {arch}  lr={cfg['lr']}  bs={cfg['train_batch_size']}  epochs={max_epoch}  K={cfg.get('num_instances', 4)} ===")
    !{cmd}
    render_run(save_dir)

In [8]:
for name, cfg in SECTION_1_MODELS.items():
    run_experiment(name, cfg, question="Q1")



=== swin_t_custom  swin_t_custom  lr=3.5e-4  bs=128  epochs=60  K=4 ===
^C


FileNotFoundError: No runs found under logs\2604_2250\Q1\swin_t_custom-veri_*

# Compare Section 1 Runs

In [ ]:
compare_runs(
    [f"logs/{RUNTIME}/Q1/{experiment}-veri" for experiment in SECTION_1_MODELS],
    runtime=RUNTIME,
    html=True,
)


# Section 2: Dataset preparation and augmentation experiment (25 marks)

Begin with the default data augmentation setting with random, horizontal flip and
Random2DTranslation.
1. Further append on top two additional data augmentation techniques, one at a time
e.g., Default + “crop”, Default + “horizontal flip”, and Default + “blurring”. Compare
the results with the default configuration in the provided code and discuss the
differences in performance. (20 marks)
2. Combine the augmentation techniques from Question 1 to find the best-performing
combination. Highlight any improvement or drop in the overall score. (5 marks)

## Q2 — augmentation experiments on a chosen model

Pick one model from Q1 and try each augmentation flag (`--crop-aug`, `--blur-aug`) plus their combination. Reuses `run_experiment` from above; per-experiment augmentation flags go through `extra_args`.

In [ ]:
Q2_CHOSEN_ARCH = "resnet50_fc512"

# Augmentation ablation for resnet50_fc512 on VeRi-776.
# Baseline = Q1 resnet50_fc512 (no augmentation).
# _crop:       crop augmentation only        (TransReID §4.2, Li et al. §3)
# _erase:      random erasing only           (BoT §3.2, TransReID §4.2, Li et al. §3, GLSIPNet §4)
# _bot_recipe: full CNN recipe               (BoT §3.2-3.3 + TransReID §4.2 + Li et al. §3)
Q2_AUGMENTATIONS = {
    f"{Q2_CHOSEN_ARCH}_crop":       ("--crop-aug",),
    f"{Q2_CHOSEN_ARCH}_erase":      ("--random-erase",),
    f"{Q2_CHOSEN_ARCH}_bot_recipe": ("--random-erase", "--crop-aug", "--label-smooth"),
}


In [ ]:
for name, cfg in Q2_AUGMENTATIONS.items():
    run_experiment(name, cfg, question="Q2")


### Compare Q2 (baseline + augmentations)

In [ ]:
!ls -l logs/2604_1855

In [ ]:
compare_runs(
    [f"logs/{RUNTIME}/Q1/{Q2_CHOSEN_ARCH}-veri"]
    + [f"logs/{RUNTIME}/Q2/{name}-veri" for name in Q2_AUGMENTATIONS],
    runtime=RUNTIME,
    html=True,
)


# Section 3: Exploration of Hyperparameters (25 marks)

Start with the default learning rate (LR) and batch size (BS).
1. Exploration of Learning Rate (LR). (10 marks)
a. Experiment with 4 values of LR (in addition to the default value).
b. Discuss the observed impact of each value on overall performance.
2. Exploration Batch sizes. (10 marks)
a. Fixing the best LR value from the experiments in question 1
above, test 4 different values of the BS in addition to the default
value.
b. Discuss the impact observed on overall performance.
3. Exploration of the optimizer. (5 marks)
3. Fixing the best Learning Rate value and best Batch Size value from the
experiments in Questions 1 and 2, respectively, test with changing the
optimizer to SGD, using PyTorch’s internal class.
a. Discuss the impact observed on overall performance

## Q3 — hyperparameter sweeps on a chosen model

Three sub-sweeps, run sequentially:
1. **LR sweep** at default batch size.
2. **Batch-size sweep** at the best LR from step 1 (set `BEST_LR` manually after viewing the comparison).
3. **Optimizer change** to SGD at the best LR + best BS (set `BEST_BS`).

All runs use `run_experiment` so any other knob in `BASE_CFG` can be added the same way.

In [ ]:
Q3_CHOSEN_ARCH = "resnet50_fc512"

def _slug(v):
    """Filesystem-safe slug for floats and ints (e.g. 0.0003 -> "0p0003")."""
    return str(v).replace(".", "p").replace("-", "n")


### Q3.1 — Learning-rate sweep

In [ ]:
Q3_LR_VALUES = ["0.0001", "0.0003", "0.001", "0.005", "0.01"]

Q3_LR_SWEEP = {
    f"{Q3_CHOSEN_ARCH}_lr_{_slug(lr)}": {"arch": Q3_CHOSEN_ARCH, "lr": lr}
    for lr in Q3_LR_VALUES
}

for name, cfg in Q3_LR_SWEEP.items():
    run_experiment(name, cfg, question="Q3")


In [ ]:
compare_runs(
    [f"logs/{RUNTIME}/Q3/{name}-veri" for name in Q3_LR_SWEEP],
    runtime=RUNTIME,
    html=True,
)


### Q3.2 — Batch-size sweep at best LR

Set `BEST_LR` to the winner from Q3.1 before running.

In [ ]:
BEST_LR = "0.0003"  # <-- update after Q3.1

Q3_BS_VALUES = [16, 32, 64, 128, 256]

Q3_BS_SWEEP = {
    f"{Q3_CHOSEN_ARCH}_lr{_slug(BEST_LR)}_bs{bs}": {
        "arch": Q3_CHOSEN_ARCH, "lr": BEST_LR, "train_batch_size": bs
    }
    for bs in Q3_BS_VALUES
}

for name, cfg in Q3_BS_SWEEP.items():
    run_experiment(name, cfg, question="Q3")


In [ ]:
compare_runs(
    [f"logs/{RUNTIME}/Q3/{name}-veri" for name in Q3_BS_SWEEP],
    runtime=RUNTIME,
    html=True,
)


### Q3.3 — SGD at best LR + best BS

Set `BEST_BS` to the winner from Q3.2.

In [ ]:
BEST_BS = 128  # <-- update after Q3.2

Q3_OPTIM_SWEEP = {
    f"{Q3_CHOSEN_ARCH}_sgd_lr{_slug(BEST_LR)}_bs{BEST_BS}": {
        "arch": Q3_CHOSEN_ARCH,
        "lr": BEST_LR,
        "train_batch_size": BEST_BS,
        "optim": "sgd",
    },
}

for name, cfg in Q3_OPTIM_SWEEP.items():
    run_experiment(name, cfg, question="Q3")


In [ ]:
compare_runs(
    [
        f"logs/{RUNTIME}/Q1/{Q3_CHOSEN_ARCH}-veri",  # original AMSGrad baseline
        *[f"logs/{RUNTIME}/Q3/{name}-veri" for name in Q3_OPTIM_SWEEP],
    ],
    runtime=RUNTIME,
    html=True,
)


In [ ]:
NOTEBOOK_END_TIME = datetime.now()
NOTEBOOK_RUNTIME = (NOTEBOOK_END_TIME - NOTEBOOK_START_TIME).total_seconds()
print(f"Notebook Run time: {NOTEBOOK_RUNTIME:.2f} seconds")

Student ID:<your id>
Student name:<your name>
UUID:66fde6ea-dd5e-483e-b809-315ce899a752
Experiment time:2026-04-26 22:50:41
Args:Namespace(root='C:\\Users\\soura\\Code\\2026\\reid\\data', source_names=['veri'], target_names=['veri'], workers=8, split_id=0, height=224, width=224, train_sampler='RandomIdentitySampler', data_fraction=0.01, random_erase=False, color_jitter=False, color_aug=False, crop_aug=False, blur_aug=False, optim='amsgrad', lr=0.00035, weight_decay=0.0005, momentum=0.9, sgd_dampening=0, sgd_nesterov=False, rmsprop_alpha=0.99, adam_beta1=0.9, adam_beta2=0.999, max_epoch=60, start_epoch=0, train_batch_size=128, test_batch_size=100, lr_scheduler='multi_step_warmup', stepsize=[20, 40], gamma=0.1, warmup_epochs=10, label_smooth=False, margin=0.3, num_instances=4, lambda_xent=1, lambda_htri=1, arch='swin_t_custom', experiment='swin_t_custom', no_pretrained=False, load_weights='', evaluate=False, eval_freq=1, start_eval=0, test_size=800, query_remove=True, patience=10, print_

C:\Users\soura\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
C:\Users\soura\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
C:\Users\soura\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
C:\Users\soura\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
C:\Users\soura\AppData\Local\Python\pythoncore-3.14-64\Lib\site-pack